# Ross Sea section — MESACLIP piControl
Monthly temperature, salinity, ideal age and σ2 along a cross-shelf section in the Ross Sea, interpolated from the
0.1° POP grid and saved for `ross_section_movie.ipynb`.

In [ ]:
import os
from pathlib import Path

import cmocean.cm as cmo
import gsw
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import xesmf
from dask.distributed import Client, LocalCluster

In [ ]:
# One thread per worker: HDF5 reads don't run in parallel across threads
cluster = LocalCluster(n_workers=len(os.sched_getaffinity(0)), threads_per_worker=1)
client = Client(cluster)
client

## Settings

In [ ]:
# Input: monthly POP output
MONTHLY_DATA_DIR = Path.home() / "MESACLIP_piControl_500year_monthly"
YEARS = (420, 519)   # model years kept: the last 100 years of the run

# Output
SECTION_FILE = "../data/Ross_section.zarr"

# Region read from the model and shown in the check maps
REGION_LAT = (-79, -66)   # °N
REGION_LON = (150, 215)   # °E

# Section: straight line in lon/lat from the Ross Ice Shelf front, across the shelf break, into the deep ocean
SECTION_START    = (178.0, -77.6)   # (°E, °N)
SECTION_END      = (190.0, -68.5)
N_SECTION_POINTS = 400              # ~2.7 km apart, finer than the model grid here (2.3–4.7 km)

# Month shown in the check plots; POP stamps each monthly mean at the start of the next month (July 519 → 0519-08-01)
CHECK_DATE = "0519-08-01"

## Open the region

In [ ]:
def pop_files(var):
    """Monthly files for one POP variable (matches the exact name, so TEMP doesn't also match TEND_TEMP)."""
    return sorted(MONTHLY_DATA_DIR.glob(f"*.pop.h.{var}.0*.nc"))


def index_box(grid, lat_range, lon_range):
    """nlat/nlon slices covering a lat/lon range. This far south, POP rows follow latitude and columns longitude.
    TLAT/TLONG are -1 on POP's removed land blocks, so those are ignored."""
    valid = grid.TLAT != -1
    row_lat = grid.TLAT.where(valid).median("nlon")
    rows = np.flatnonzero(((row_lat >= lat_range[0]) & (row_lat <= lat_range[1])).values)
    col_lon = grid.TLONG.where(valid).isel(nlat=rows).median("nlat")   # only the chosen rows (the grid bends further north)
    cols = np.flatnonzero(((col_lon >= lon_range[0]) & (col_lon <= lon_range[1])).values)
    return dict(nlat=slice(rows[0], rows[-1] + 1), nlon=slice(cols[0], cols[-1] + 1))


# Grid (static) read once; the monthly files then only need the data variable
first_temp_file = xr.open_dataset(pop_files("TEMP")[0], decode_timedelta=True)
grid_vars = [name for name in first_temp_file.variables
             if name not in first_temp_file.dims and "time" not in first_temp_file[name].dims]
region = index_box(first_temp_file, REGION_LAT, REGION_LON)
grid = first_temp_file.isel(region)


def open_pop_monthly(var):
    """Lazily open one monthly POP variable over the region and YEARS."""
    data = xr.open_mfdataset(
        pop_files(var), combine="nested", concat_dim="time",
        data_vars="minimal", coords="minimal", compat="override", join="override",
        drop_variables=grid_vars + ["time_bound"],
        chunks={"time": 12, "z_t": -1, "nlat": 267, "nlon": 400},   # matches on-disk chunks
        decode_timedelta=True, parallel=True, engine="h5netcdf",
    )[var].isel(region)
    return data.sel(time=slice(f"{YEARS[0]:04d}-02-01", f"{YEARS[1] + 1:04d}-01-01"))   # stamps of Jan YEARS[0] to Dec YEARS[1]


ds = xr.Dataset({var: open_pop_monthly(var) for var in ("TEMP", "SALT", "IAGE")})
ds = ds.assign_coords(lat=grid.TLAT, lon=grid.TLONG, depth=ds.z_t / 100)   # depth in m

# σ2 (TEOS-10): POP gives practical salinity and potential temperature
pressure = gsw.p_from_z(-ds.depth, ds.lat)
absolute_salinity = gsw.SA_from_SP(ds.SALT, pressure, ds.lon, ds.lat)
ds["SIGMA2"] = gsw.sigma2(absolute_salinity, gsw.CT_from_pt(absolute_salinity, ds.TEMP))
ds

## Check maps: ocean depth and bottom σ2, with the section line

In [ ]:
section_points = xr.Dataset(coords={
    "lon": ("locations", np.linspace(SECTION_START[0], SECTION_END[0], N_SECTION_POINTS)),
    "lat": ("locations", np.linspace(SECTION_START[1], SECTION_END[1], N_SECTION_POINTS)),
})

ocean_depth = (grid.HT / 100).where(grid.HT > 0)   # m
# Bottom σ2: fill each level down from above, so the last level holds the deepest ocean value
bottom_sigma2 = ds.SIGMA2.sel(time=CHECK_DATE).squeeze("time").ffill("z_t").isel(z_t=-1).compute()

fig, axes = plt.subplots(2, 1, figsize=(10, 9), sharex=True)
ocean_depth.plot.pcolormesh(ax=axes[0], x="TLONG", y="TLAT", cmap="Blues", vmax=4000,
                            cbar_kwargs=dict(label="depth (m)"))
bottom_sigma2.plot.pcolormesh(ax=axes[1], x="lon", y="lat", cmap=cmo.dense, vmin=37.1, vmax=37.4,
                              cbar_kwargs=dict(label=r"bottom $\sigma_2$ (kg m$^{-3}$)"))
for ax in axes:
    ocean_depth.plot.contour(ax=ax, x="TLONG", y="TLAT", levels=[500, 1000, 2000, 3000], colors="k", linewidths=0.6)
    ax.plot(section_points.lon, section_points.lat, "r-", lw=2)
    ax.set(xlim=REGION_LON, ylim=REGION_LAT)
axes[0].set_xlabel("")
axes[1].set_title(f"Bottom $\\sigma_2$, {CHECK_DATE}")
plt.tight_layout()

## Interpolate onto the section
Bilinear interpolation (xesmf) from the model points around the section. The model's ocean depth `HT` is
interpolated too, as a smooth seafloor for the movie.

In [ ]:
margin = 0.5   # degrees around the section
section_box = index_box(grid,
                        (float(section_points.lat.min()) - margin, float(section_points.lat.max()) + margin),
                        (float(section_points.lon.min()) - margin, float(section_points.lon.max()) + margin))
near_section = ds.isel(section_box).chunk(nlat=-1, nlon=-1)   # xesmf needs one horizontal chunk

valid = near_section.lat != -1   # POP lat/lon are -1 on removed land blocks
regridder = xesmf.Regridder(near_section.assign_coords(lat=near_section.lat.where(valid), lon=near_section.lon.where(valid)),
                            section_points, "bilinear", locstream_out=True, unmapped_to_nan=True, periodic=False)
section = regridder(near_section[["IAGE", "TEMP", "SALT", "SIGMA2"]], keep_attrs=True, skipna=True, na_thres=0.5)
section["seafloor_depth"] = regridder(grid.HT.isel(section_box) / 100)   # m
section

## Check section

In [ ]:
check_month = section.sel(time=CHECK_DATE).squeeze("time").compute()
sigma2_levels = np.r_[[36.6, 36.7, 36.8, 36.9], np.arange(37.0, 37.4, 0.02)]

fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
check_month.IAGE.plot.contourf(ax=axes[0], x="lat", y="depth", yincrease=False, cmap="viridis", levels=12,
                               vmin=0, cbar_kwargs=dict(label="Ideal age (years)"))
check_month.SIGMA2.plot.contourf(ax=axes[1], x="lat", y="depth", yincrease=False, cmap=cmo.dense, levels=sigma2_levels,
                                 cbar_kwargs=dict(label=r"$\sigma_2$ (kg m$^{-3}$)"))
for ax in axes:
    ax.fill_between(check_month.lat, check_month.seafloor_depth, 4000, color="0.4")
    ax.set_ylim(4000, 0)
    ax.set_ylabel("depth (m)")
axes[0].set_title(f"Ross Sea section, {CHECK_DATE}")
axes[0].set_xlabel("")
axes[1].set_title("")
axes[1].set_xlabel("latitude")
plt.tight_layout()

## Save

In [ ]:
section.to_zarr(SECTION_FILE, mode="w")